<a href="https://colab.research.google.com/github/monsterdevgit/bluechip_hackathon/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **INSTALL & IMPORTS**

In [30]:
%pip install -r requirements.txt

In [32]:
import os
import weaviate
import pandas as pd

from helpers.memory import update_memory, user_profile
from helpers.planner import planner
from helpers.retrieval import retrieve_products
from helpers.reranker import rerank
from helpers.generator import generate_response

from weaviate.auth import AuthApiKey
from weaviate.util import generate_uuid5
import weaviate.classes.config as wc
import weaviate.classes.query as wq

from google.colab import userdata
import google.generativeai as genai

### **GEMINI & WEAVIATE SETUP**

In [33]:
genai.configure(api_key=userdata.get('API-Key'))

model = genai.GenerativeModel("gemini-2.5-flash")

In [34]:
WEAVIATE_URL = userdata.get("WEAVIATE-url")
WEAVIATE_API_KEY = userdata.get("WEAVIATE")

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY),
)

print(client.is_live())

True


### **Dataset Processing**

In [35]:
# Load the dataset
file_path = "/content/dataset/Amazon-Products.csv"
dataset = pd.read_csv(file_path)
# Display the first few rows of the dataset
dataset.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/31UISB90sY...,https://www.amazon.in/Lloyd-Inverter-Convertib...,4.2,"2,255","₹32,999","₹58,990"
1,1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.2,"2,948","₹46,490","₹75,990"
2,2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Inverter-Convertible-...,4.2,"1,206","₹34,490","₹61,990"
3,3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.0,69,"₹37,990","₹68,990"
4,4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/41lrtqXPiW...,https://www.amazon.in/Carrier-Inverter-Split-C...,4.1,630,"₹34,490","₹67,790"


##### **Dataset cleaning**

In [36]:
# Check for the missing and duplicates
print(f"\nUncleaned DataFrame Rows: {dataset.shape[0]}")
print(f"\nAll of the Duplicated Rows: {dataset.duplicated().sum()}" )
print(f"\nAll of the N/A Rows: {dataset.isna().sum()}" )


Uncleaned DataFrame Rows: 551585

All of the Duplicated Rows: 0

All of the N/A Rows: Unnamed: 0             0
name                   0
main_category          0
sub_category           0
image                  0
link                   0
ratings           175794
no_of_ratings     175794
discount_price     61163
actual_price       17813
dtype: int64


In [37]:
# Handling missing values
df_cleaned = dataset.dropna()

# Display the cleaned DataFrame
print(f"\nCleaned DataFrame Rows: {df_cleaned.shape[0]}")


Cleaned DataFrame Rows: 340680


In [38]:
def conversion(amount, exchange_rate=19):
    """
    Convert INR to Naira.
    Returns:
    str: Amount in Naira (₦).
    """
    amount_in_inr = float(amount.replace('₹', '').replace(',', ''))
    amount_in_ngn = amount_in_inr * exchange_rate
    return f"₦{amount_in_ngn:.2f}"

df_cleaned['actual_price'] = df_cleaned['actual_price'].apply(conversion)
df_cleaned['discount_price'] = df_cleaned['discount_price'].apply(conversion)

df_cleaned.head()

/tmp/ipykernel_20131/3543625835.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['actual_price'] = df_cleaned['actual_price'].apply(conversion)
/tmp/ipykernel_20131/3543625835.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['discount_price'] = df_cleaned['discount_price'].apply(conversion)


,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/31UISB90sY...,https://www.amazon.in/Lloyd-Inverter-Convertib...,4.2,"2,255",₦626981.00,₦1120810.00
1,1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.2,"2,948",₦883310.00,₦1443810.00
2,2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Inverter-Convertible-...,4.2,"1,206",₦655310.00,₦1177810.00
3,3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.0,69,₦721810.00,₦1310810.00
4,4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/41lrtqXPiW...,https://www.amazon.in/Carrier-Inverter-Split-C...,4.1,630,₦655310.00,₦1288010.00


In [40]:
print(f"datatypes before\n:{df_cleaned.dtypes}")
def clean_price(price):
 # Remove currency symbol and commas, then convert to float
 return float(price.replace('₦', '').replace(',', ''))
df_cleaned['discount_price'] = df_cleaned['discount_price'].apply(clean_price)
df_cleaned['actual_price'] = df_cleaned['actual_price'].apply(clean_price)
df_cleaned['ratings'] = pd.to_numeric(df_cleaned['ratings'], errors='coerce')
df_cleaned['no_of_ratings'] = pd.to_numeric(df_cleaned['no_of_ratings'], errors='coerce')
print(f"datatypes after\n:{df_cleaned.dtypes}")
print(f"\nCleaned and Transformed DataFrame Rows: {df_cleaned.shape[0]}")

datatypes before
:Unnamed: 0         int64
name              object
main_category     object
sub_category      object
image             object
link              object
ratings           object
no_of_ratings     object
discount_price    object
actual_price      object
dtype: object


/tmp/ipykernel_20131/367141170.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['discount_price'] = df_cleaned['discount_price'].apply(clean_price)
/tmp/ipykernel_20131/367141170.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['actual_price'] = df_cleaned['actual_price'].apply(clean_price)


datatypes after
:Unnamed: 0          int64
name               object
main_category      object
sub_category       object
image              object
link               object
ratings           float64
no_of_ratings     float64
discount_price    float64
actual_price      float64
dtype: object

Cleaned and Transformed DataFrame Rows: 340680


/tmp/ipykernel_20131/367141170.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['ratings'] = pd.to_numeric(df_cleaned['ratings'], errors='coerce')
/tmp/ipykernel_20131/367141170.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['no_of_ratings'] = pd.to_numeric(df_cleaned['no_of_ratings'], errors='coerce')


### **Cluster Vector Database**

In [41]:
# Delete the previous Products collection and all its data from Weaviate
client.collections.delete("Products")
print("Old collection removed")

Old collection removed


##### **Create Products Collection in the cluster**

In [42]:
properties = [
    wc.Property(name="name", data_type=wc.DataType.TEXT),
    wc.Property(name="main_category", data_type=wc.DataType.TEXT),
    wc.Property(name="sub_category", data_type=wc.DataType.TEXT),

    # Non-searchable / no embedding fields
    wc.Property(name="image", data_type=wc.DataType.TEXT, skip_vectorization=True),
    wc.Property(name="link", data_type=wc.DataType.TEXT, skip_vectorization=True),

    # Numeric fields (no vectorization needed)
    wc.Property(name="ratings", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="no_of_ratings", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="discount_price", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="actual_price", data_type=wc.DataType.NUMBER, skip_vectorization=True),
]

# Create collection
try:
    client.collections.create(
        name="Products",
        properties=properties,
        vectorizer_config=None
    )
    print("Product collection created successfully.")

except Exception as e:
    print(f"Failed to create Product collection: {e}")

Product collection created successfully.


 ##### **Insert Products to Vector Database**

In [43]:
# Check if the collection is created in our cluster.
# Since the dataset is large, with approximately ~>550,000 entries, we will filter it
# Specifically to include only appliances products and then sample 1000 products from this filtered subset.

df_filtered = df_cleaned[df_cleaned['main_category'] == 'appliances']
df_sampled = df_filtered.sample(n = 1000)

In [44]:
def clean_float(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except:
        return None

In [45]:
products = client.collections.get("Products")
# Enter context manager
with products.batch.dynamic() as batch:
    for i, product in tqdm(df_sampled.iterrows(), total=df_sampled.shape[0]):

        product_obj = {
            "name": str(product.get("name", "")),
            "main_category": str(product.get("main_category", "")),
            "sub_category": str(product.get("sub_category", "")),
            "image": str(product.get("image", "")),
            "link": str(product.get("link", "")),
            "ratings": clean_float(product.get("ratings")),
            "no_of_ratings": clean_float(product.get("no_of_ratings")),
            "discount_price": clean_float(product.get("discount_price")),
            "actual_price": clean_float(product.get("actual_price")),
        }

        batch.add_object(
            properties=product_obj,
            uuid=generate_uuid5(product_obj["link"])
        )

100%|██████████| 1000/1000 [00:00<00:00, 3582.16it/s]


### **AGENT PIPELINE**

In [46]:
def agent_chat(user_input):

    # 1. memory
    profile = update_memory(model, user_input)

    # 2. planning
    plan = planner(model, user_input, profile)

    # 3. retrieval
    items = retrieve_products(
        products,
        plan["search_query"],
        plan.get("category")
    )

    # 4. rerank
    ranked = rerank(model, user_input, items)

    # 5. generate
    reply = generate_response(model, user_input, ranked, profile)

    return reply, ranked[:5]

In [47]:
while True:

    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response, items = agent_chat(user_input)

    print("\nAssistant:\n", response)

    print("\n🛒 Products:\n")

    for i, item in enumerate(items):
        print(f"{i}. {item.properties['name']} - {item.properties['discount_price']}")
        print("-" * 50)

    # FEEDBACK INPUT
    choice = input("\nChoose a product you like (number) or press Enter to skip: ")

    if choice.isdigit():
        idx = int(choice)
        if 0 <= idx < len(items):
            selected_product = items[idx].properties["name"]
            store_feedback(selected_product)
            print(f"Added to feedback: {selected_product}")

You: hey

Assistant:
 Hey there!

I've got a few cool products here. Since you didn't ask for anything specific, how about I tell you about a couple of different things?

*   We have the **Daikin 1.5 Ton 3 Star Inverter Split AC** – it's got a PM 2.5 Filter and some neat features like Dew Clean Technology for fresh air.
*   Or, if you're looking for something for the kitchen, there's the **TDW Hand Mixer Easy Mix**, a powerful electric beater with 7 speeds that could be super handy.
*   We also have the **Samsung 183 L 2 Star Inverter Direct-Cool Single Door Refrigerator** in a snazzy Mystic Overlay Red if you need a new fridge!

Do any of those sound interesting, or were you looking for something else entirely? Let me know!

🛒 Products:

0. Daikin 1.5 Ton 3 Star Inverter Split AC (Copper, PM 2.5 Filter, Triple Display, Dew Clean Technology, Coanda Airflow, MTKL... - 750481.0
--------------------------------------------------
1. The Quasar Store Mixer Grinder Chutney Jar (500 ml) Mixer

KeyboardInterrupt: Interrupted by user